# Water Risk

This notebook loads the baseline annual Aqueduct 4.0 water risk data and normalizes the column containing total water risk with default weighting, raw numbers was normalized. Finally, the resulting layer is rasterized using the bii layer as reference grid and than overlayed on country boundaries and creates:
- a raster (with bii as reference raster)
- a map figure (`OUT_PNG`)

## How to run
1. Put the required input files in the same folder as this notebook (or edit the paths in the **Configuration** cell below).
2. Run the cells from top to bottom.

## Required files
- `aqueduct_tot.gpkg`
- `bii_5000m.tif`
- `World_Countries_(Generalized)_8414823838130214587.gpkg` (or your country layer)


In [ ]:
# Configuration (edit these paths / settings)
AQUEDUCT_TOT_GPKG = 'aqueduct_tot.gpkg'
BII_5000M_TIF = 'sensitivity/biodiversity_intactness/bii_5000m.tif'
WATER_RISK_RASTERIZED_TIF = 'water_risk_rasterized.tif'
WORLD_COUNTRIES_GENERALIZED_8414823838130214587_GPKG = 'World_Countries_(Generalized)_8414823838130214587.gpkg'
WATER_RISK_5000M_TIF = 'water_risk_5000m.tif'

# Tip: keep data files out of the repo (use .gitignore) or use Git LFS/DVC for large files.


In [ ]:
#import packages
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.transform import from_origin
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from rasterio.enums import Resampling
from rasterio.features import geometry_mask
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
from rasterio.windows import Window
from rasterio.features import rasterize
import shapely.geometry as sg
import shapely.ops as ops

In [ ]:
#load gpkg with aqueduct data
gdf = gpd.read_file(AQUEDUCT_TOT_GPKG)

In [ ]:
#normalize annual baseline total water risk with default weighting, raw numbers: for w_awr_def_tot_raw
col = "w_awr_def_tot_raw"
nodata_val = -9999.0 #define no data value

mask = gdf[col] != nodata_val
valid = gdf.loc[mask, col]

min_val = valid.min()
max_val = valid.max()

gdf["tot_norm"] = np.nan
gdf.loc[mask, "tot_norm"] = (valid - min_val) / (max_val - min_val)


In [ ]:
#clean geometry
gdf_clean = gdf[~gdf.geometry.is_empty].copy()


In [ ]:
# Create vertical LineStrings representing the antimeridian
# +180° longitude (right edge of geographic CRS)
antimeridian = sg.LineString([(180, -90), (180, 90)])

# -180° longitude (left edge of geographic CRS)
# Numerically different from +180, but geographically the same line
antimeridian_neg = sg.LineString([(-180, -90), (-180, 90)])


def split_dateline(geom):
    """    Splits geometries that cross the ±180° longitude line (antimeridian).
    This prevents polygons from wrapping incorrectly across the globe
    """
    try:
        # If geometry crosses +180°, split it there
        if geom.crosses(antimeridian):
            geom = ops.split(geom, antimeridian)

        # If geometry crosses -180°, split it there
        # (needed because some geometries may be represented using -180)
        if geom.crosses(antimeridian_neg):
            geom = ops.split(geom, antimeridian_neg)

        return geom

    # If splitting fails (e.g., invalid geometry), return original geometry
    except Exception:
        return geom


# Apply the splitting function to every geometry in the GeoDataFrame
gdf_clean["geometry"] = gdf_clean.geometry.apply(split_dateline)


# If splitting created MultiPolygons or GeometryCollections,
# explode them so each part becomes its own row
# (important for rasterization and spatial analysis)
gdf_clean = gdf_clean.explode(index_parts=False)


# Remove invalid results:
# - geometries that are None
# - geometries that are empty after splitting
gdf_clean = gdf_clean[
    gdf_clean.geometry.notna() & ~gdf_clean.geometry.is_empty
].copy()


In [ ]:
# rasterize water risk variable using bii as reference raster

#paths
ref = BII_5000M_TIF #reference raster
out_path = WATER_RISK_RASTERIZED_TIF

# Open reference raster as the template grid
with rasterio.open(ref) as src:
    profile = src.profile.copy()
    crs = src.crs
    transform = src.transform
    out_shape = (src.height, src.width)

    # reproject polygons to match raster CRS
    gdf_r = gdf_clean.to_crs(crs)

    b = gdf_r.geometry.bounds
    w = b.maxx - b.minx
    h = b.maxy - b.miny
    aspect = (w / h).where(w > h, h / w)
    
    # keep only sane geometries for clean reprojection
    gdf_r = gdf_r[aspect < 200].copy()
    
    
    # Write output windowed processing
    with rasterio.open(out_path, "w", **profile) as dst:
        for block_index, window in src.block_windows(1):    
            win_transform = rasterio.windows.transform(window, transform)
        
            window_raster = rasterize(
                    [(geom, value) for geom, value in zip(gdf_r.geometry, gdf_r.tot_norm)],
                    out_shape=(window.height, window.width),  
                    transform=win_transform,
                    fill=-9999,
                    dtype="float32",
                    all_touched=True
                )
    
            dst.write(window_raster, 1, window=window)

print("Saved:", out_path)

In [ ]:
#add country boundaries

#paths
water_risk_path = WATER_RISK_RASTERIZED_TIF  # your existing raster
countries_path = WORLD_COUNTRIES_GENERALIZED_8414823838130214587_GPKG
out_path = WATER_RISK_5000M_TIF

# Open water risk raster as the template grid
with rasterio.open(water_risk_path) as src:
    profile = src.profile.copy()
    crs = src.crs
    transform = src.transform

    # Read countries and project to raster CRS
    world = gpd.read_file(countries_path).to_crs(crs)

    # Write output windowed processing
    with rasterio.open(out_path, "w", **profile) as dst:
            for block_index, window in src.block_windows(1):    
                water_risk = src.read(1, window=window).astype("float32")
                win_transform = rasterio.windows.transform(window, transform)
        
                country_mask = rasterize(
                    [(geom, 1) for geom in world.geometry],
                    out_shape=(window.height, window.width),
                    transform=win_transform,
                    fill=0,
                    dtype="uint8"
                )
    
    
                # Create output for this window
                out = np.full((window.height, window.width), -9999, dtype="float32")
                inside = (country_mask == 1)
                out[inside] = water_risk[inside]
    
                dst.write(out, 1, window=window)

print("Saved:", out_path)


In [ ]:
#plot

# paths
raster_path = WATER_RISK_5000M_TIF
countries_path = WORLD_COUNTRIES_GENERALIZED_8414823838130214587_GPKG
out_png = "water_risk.png"

max_width = 5000

#open raster
with rasterio.open(raster_path) as src:
    bounds = src.bounds
    crs = src.crs
    nodata_val = src.nodata

    #calculate display size
    scale = max_width / src.width
    out_w = int(src.width * scale)
    out_h = int(src.height * scale)

    #read raster band with new size using .nearest resampling
    arr = src.read(
        1,
        out_shape=(out_h, out_w),
        resampling=Resampling.nearest
    ).astype("float32")

    #adjust transform to new size
    transform = src.transform * src.transform.scale(
        src.width / out_w,
        src.height / out_h
    )

#open country boundaries gpkg and drop Antarctica 
world = gpd.read_file(countries_path).to_crs(crs)
world = world[world["COUNTRY"] != "Antarctica"].copy()


#build country mask
country_mask = geometry_mask(
    geometries=world.geometry,
    transform=transform,
    invert=True,
    out_shape=arr.shape
)

#define nodata
nodata = (arr == -9999)

#convert -9999 to nan for plotting
arr = arr.astype("float32")
arr[arr == -9999] = np.nan

#mask data outside country and no data for arr
masked = np.ma.masked_where((~country_mask) | nodata, arr)

#transparency settings
alpha = np.where(country_mask, 0.95, 0.0)

#define valid values
valid = arr[country_mask & np.isfinite(arr)]

# robust min/max from valid pixels
vmin = float(valid.min())
vmax = float(valid.max())

nodata_color = "#D1D5DB"
cmap = LinearSegmentedColormap.from_list(
    "water_risk_red_grad",
    ["#FFF1F2", "#EF4444", "#7F1D1D"]
)
cmap.set_bad(color=nodata_color)

#build figure, set size and background color
fig, ax = plt.subplots(figsize=(14, 7), dpi=200)
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

#plot country boundaries, set color and linewidth
world.plot(
    ax=ax,
    facecolor="#F6F7F9",
    edgecolor="#B9C0C8",
    linewidth=0.35,
    zorder=1
)

# plot raster 
raster = ax.imshow(
    masked,
    cmap=cmap,
    vmin=vmin,
    vmax=vmax,
    extent=[bounds.left, bounds.right, bounds.bottom, bounds.top],
    interpolation="nearest",
    alpha=alpha,          # <-- outside countries becomes white
    zorder=2
)

# Plot country boundaries on top
world.boundary.plot(ax=ax, color="#695C5A", linewidth=0.3, zorder=2)

#set title
ax.set_title("Water Risk", fontsize=18, fontweight="semibold", pad=14)

#no axis
ax.set_axis_off()

#create colorbar
cbar = plt.colorbar(raster, ax=ax, fraction=0.03, pad=0.02)#width of color bar relative to plot and space between plot and colorbar
cbar.set_label("Water risk score", fontsize=11, color="#2B2F36")
cbar.ax.tick_params(labelsize=10, colors="#2B2F36")#numbers on colorbar
cbar.outline.set_edgecolor("#E3E6EA")#edgecolor of colorbar
cbar.outline.set_linewidth(1.0)
cbar.ax.set_facecolor("white")

# NoData legend patch 
legend_handles = [Patch(facecolor=nodata_color, edgecolor="none", label="No data")]
leg = ax.legend(
    handles=legend_handles,
    loc="lower left",
    frameon=True,
    framealpha=1,
    facecolor="white",
    edgecolor="#E3E6EA",
    borderpad=0.8,
    handlelength=1.2,
)

plt.savefig(out_png, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
